# Overmerge Cluster Analysis

Compare per-work centroid-similarity metrics on labeled authors (overmerged vs clean).
Uses pre-computed cosine_similarity from authorship_embedding_similarity — no raw vectors needed.

- Per-work fit score: min, p5, p10, stddev, frac below thresholds
- Job: #105.11 aer-overmerge-signal-calibration

In [ ]:
import time
from datetime import datetime

SIMILARITY_TABLE = "openalex.aer.authorship_embedding_similarity"
CHECKPOINT_TABLE = "openalex.aer.overmerge_cluster_analysis"
MIN_WORKS = 5

## Define labeled authors

In [ ]:
# Zendesk overmerge cases
zendesk_overmerged = [
    5106038276, 5127355048, 5121306932, 5071424520, 5088402891,
    5045317380, 5100726594, 5108072493, 5009709588, 5045904720,
    5012309101, 5069151214, 5011165427
]

# Gold standard splits (overmerged)
gold_splits = [
    5085619921, 5041864386, 5117075932, 5110196385, 5065096990,
    5037835036, 5071539405, 5017142455, 5110027945, 5091859986,
    5108623528, 5101092520, 5070168570, 5072305232, 5013478230,
    5009307852, 5016882470, 5091442005, 5079979698, 5108570566,
    5110358878, 5079166689, 5108748587, 5102755488, 5079652777,
    5113385911, 5113952815, 5010422511, 5007114677, 5037924045,
    5025452752, 5088152507, 5073218194, 5112257425, 5061138371,
    5033771456, 5101839347, 5074059264, 5080312909, 5010975613,
    5048151405, 5052571385, 5111372795, 5112390495, 5103459170,
    5103347958, 5062125091, 5110749239, 5021173676, 5113857253,
    5108453412, 5076836352, 5060512575, 5079413276, 5079617160,
    5055163904, 5064679635, 5109293548, 5067290858, 5050939922,
    5056229359, 5037435179, 5006626689, 5103976114, 5010484703,
    5087117526, 5019217807, 5075755572, 5109148503, 5014188192,
    5051189473, 5058421597, 5057408903, 5056945568, 5074432852,
    5019862321, 5103907138, 5003683328, 5103173829, 5051468011,
    5038742031, 5062220638, 5114098473, 5001106162, 5029904482,
    5073480387, 5002981530
]

# Gold standard confirmed (clean)
gold_confirmed = [
    5087166684, 5081747147, 5082325782, 5012168214, 5012434282,
    5003686344, 5010617958, 5084195402, 5079654941, 5000848656,
    5061377970, 5000447750, 5023844756, 5080591937, 5027129242,
    5067354901, 5109968665, 5037064192, 5072647734, 5016361065,
    5026928687, 5002615359, 5027008194, 5063093897, 5108260432,
    5011946452, 5079497159, 5083508133, 5109604058, 5079798553,
    5088931673, 5060557374, 5086191294, 5044638100, 5036926833,
    5037676122, 5082406391, 5073380969, 5019534047, 5090821911,
    5111962545, 5027442102, 5036629892, 5084506603, 5075560544,
    5114030225, 5047130741, 5088150711, 5024736221, 5046646750,
    5111945307, 5066075206, 5063684507, 5053820618, 5055178909,
    5087090962, 5103243948, 5050549195, 5033358050, 5102850112,
    5055945041, 5026328537, 5022112939, 5011576899, 5036961392,
    5000185973, 5013710631, 5088159296, 5063641432, 5018449545,
    5074749021, 5052625144, 5078623160, 5030659236, 5004860476,
    5002085848, 5088382884, 5050861768, 5070669440, 5035638193,
    5027340715, 5067395854, 5018332726, 5000163809, 5048092761,
    5063197873, 5077040304, 5033702363, 5110246466, 5074731779,
    5015418171, 5052638321, 5085326237, 5061312215, 5075570397,
    5112171421, 5050680576, 5017797591, 5065396267, 5087110857
]

overmerged_set = set(zendesk_overmerged + gold_splits)
clean_set = set(gold_confirmed)
all_authors = list(overmerged_set | clean_set)

print(f"Overmerged: {len(overmerged_set)}, Clean: {len(clean_set)}, Total: {len(all_authors)}")

## Compute per-author metrics from pre-computed cosine_similarity (no vectors)

In [ ]:
author_ids_str = ','.join(str(a) for a in all_authors)

t0 = time.time()
print(f"Pulling per-work cosine_similarity for {len(all_authors)} authors...")

# All computation in SQL — no vectors, just scalars
spark.sql(f"DROP TABLE IF EXISTS {CHECKPOINT_TABLE}")

spark.sql(f"""
    CREATE TABLE {CHECKPOINT_TABLE} AS
    WITH per_author AS (
        SELECT
            author_id,
            COUNT(*) AS work_count,
            AVG(cosine_similarity) AS mean_sim,
            MIN(cosine_similarity) AS min_sim,
            STDDEV(cosine_similarity) AS std_sim,
            PERCENTILE(cosine_similarity, 0.05) AS p5_sim,
            PERCENTILE(cosine_similarity, 0.10) AS p10_sim,
            PERCENTILE(cosine_similarity, 0.25) AS q1_sim,
            PERCENTILE(cosine_similarity, 0.75) AS q3_sim,
            SUM(CASE WHEN cosine_similarity < 0.5 THEN 1 ELSE 0 END) / COUNT(*) AS frac_below_05,
            SUM(CASE WHEN cosine_similarity < 0.6 THEN 1 ELSE 0 END) / COUNT(*) AS frac_below_06,
            SUM(CASE WHEN cosine_similarity < 0.7 THEN 1 ELSE 0 END) / COUNT(*) AS frac_below_07
        FROM {SIMILARITY_TABLE}
        WHERE author_id IN ({author_ids_str})
        GROUP BY author_id
        HAVING COUNT(*) >= {MIN_WORKS}
    )
    SELECT
        pa.*,
        CASE
            WHEN pa.author_id IN ({','.join(str(a) for a in overmerged_set)})
            THEN 'overmerged'
            ELSE 'clean'
        END AS label
    FROM per_author pa
""")

t1 = time.time()
count = spark.sql(f"SELECT COUNT(*) AS n FROM {CHECKPOINT_TABLE}").collect()[0]['n']
print(f"Done: {count} authors computed in {t1 - t0:.0f}s")

## Summary

In [ ]:
import numpy as np

rows = spark.sql(f"SELECT * FROM {CHECKPOINT_TABLE}").collect()
results = [r.asDict() for r in rows]

output_lines = []
output_lines.append(f"Authors analyzed: {len(results)}")
output_lines.append("")
output_lines.append("ALL METRICS - separation (median overmerged vs median clean):")
output_lines.append("=" * 80)

metrics = ['mean_sim', 'min_sim', 'std_sim', 'p5_sim', 'p10_sim', 'q1_sim',
           'frac_below_05', 'frac_below_06', 'frac_below_07']

for metric in metrics:
    over_vals = [r[metric] for r in results if r['label'] == 'overmerged' and r[metric] is not None]
    clean_vals = [r[metric] for r in results if r['label'] == 'clean' and r[metric] is not None]
    if over_vals and clean_vals:
        over_med = np.median(over_vals)
        clean_med = np.median(clean_vals)
        gap = abs(over_med - clean_med)
        pooled_std = np.std(over_vals + clean_vals)
        effect_size = gap / pooled_std if pooled_std > 0 else 0
        direction = 'higher' if over_med > clean_med else 'lower'
        line = f"  {metric:20s}: overmerged={over_med:.4f} clean={clean_med:.4f} gap={gap:.4f} effect={effect_size:.2f} ({direction})"
        output_lines.append(line)

output_lines.append("")
output_lines.append("PER-LABEL DETAIL:")
for label in ['overmerged', 'clean']:
    subset = [r for r in results if r['label'] == label]
    output_lines.append(f"  {label} (n={len(subset)}):")
    for metric in metrics:
        vals = [r[metric] for r in subset if r[metric] is not None]
        if vals:
            output_lines.append(f"    {metric}: mean={np.mean(vals):.4f} median={np.median(vals):.4f} q1={np.percentile(vals,25):.4f} q3={np.percentile(vals,75):.4f}")

output = "\n".join(output_lines)
print(output)
dbutils.notebook.exit(output)